# 🌊 OCEAN RESCUE – THE DEEP SEA ADVENTURE
### An interactive Data Science learning game built with Python, Pandas, NumPy, Matplotlib, Scikit-learn and IPyWidgets

> **How to run:** run the cells from top to bottom (**Kernel ▸ Restart & Run All**). The game appears under the very last game cell (🚀 *Launch*).

---
# 📘 DOCUMENTATION
## 1. Game title and introduction
**OCEAN RESCUE – THE DEEP SEA ADVENTURE** is a 2D underwater adventure. The ocean ecosystem is in danger because of pollution, abnormal water temperatures and falling marine populations. You are an underwater explorer who must explore the ocean, rescue animals, clean polluted regions and restore ocean health.

Data Science is **not a quiz** here – it is the *engine* of the game. Every number you see is generated by Python code, every graph is drawn live, and every decision you make changes the dataset, your score and the ocean-health indicator.

## 2. Game objectives
* Complete **four progressive missions** (each unlocks the next).
* Discover regions, rescue animals, clean polluted regions and make good restoration decisions.
* Raise the **Ocean Health %** (the live mean health score of all 36 regions) and earn the rank **Marine Hero**.

## 3. Learning objectives
After completing the game, learners should be able to:
1. Understand **data exploration and sampling** (and why representative samples matter).
2. Understand **classification** using environmental data.
3. Interpret **correlation** between environmental variables (and tell it apart from causation).
4. Perform **outlier detection** (IQR / Z-score) and **data visualisation**.
5. Understand **machine-learning classification** (train/test split, accuracy, confusion matrix).
6. Make **decisions using data analysis**.

## 4. Required libraries and installation
| Library | Used for |
|---|---|
| `pandas` | ocean dataset creation, cleaning and processing |
| `numpy` | numerical calculations, random data generation |
| `matplotlib` | all graphs (histograms, bar charts, heat-map, confusion matrix) |
| `scikit-learn` | Decision Tree + Random Forest classifiers, train/test split, metrics |
| `ipywidgets` | buttons, D-pad controls, sliders, drop-downs, navigation |

```bash
pip install numpy pandas matplotlib scikit-learn ipywidgets
# only needed for classic Jupyter Notebook / older JupyterLab:
jupyter nbextension enable --py widgetsnbextension
```
Works in Jupyter Notebook 6/7, JupyterLab, VS Code notebooks and Google Colab (Python 3.9+).

## 5. How to play
1. Run all cells, then use the **main menu** → **▶ Start Game**.
2. Read the learning objectives, then open the **Ocean Map** hub. Missions unlock one after another.
3. **Mission 1** – steer the submarine with the ⬆️ ⬅️ ⬇️ ➡️ buttons (every move costs oxygen), then pick the regions you want to investigate.
4. **Mission 2** – for each animal choose *Rescue now* or *Not urgent* by reading the water data.
5. **Mission 3** – use the IQR / Z-score tools to find unusual pollution and spend your limited cleanup units wisely.
6. **Mission 4** – train the ML model, predict each region's condition, choose restoration strategies (max. 6 actions).
7. Watch the HUD at the top: ⭐ score, 🫧 oxygen, 🎯 mission progress, 🌊 ocean health.

| Action | Points |
|---|---|
| Discover a new ocean region | +10 |
| Correctly identify an endangered area | +15 |
| Rescue a marine animal | +20 |
| Successfully clean a polluted region | +20 (+10 if it is a statistical outlier) |
| Successful restoration decision | +30 (+15 partly suitable) |

## 6. The four missions
| # | Mission | Gameplay | Data Science concept |
|---|---|---|---|
| 1 | 🧭 **Explore the Ocean** | Drive a submarine over a 6×6 map, discover regions, then choose a sample to investigate and compare it with the whole ocean. | Sampling & data exploration |
| 2 | 🐢 **Save the Marine Animals** | Read temperature, oxygen, plastic, quality and population data for 8 animals and decide who needs rescue now. Then see the correlation heat-map and feature importance. | Classification, feature identification, correlation ≠ causation |
| 3 | 🧹 **Pollution Detection & Cleanup** | Detect unusual pollution with a *dynamic* IQR / Z-score threshold, spend limited cleanup units, see before/after graphs. | Outlier detection, visualisation, statistical interpretation |
| 4 | 🌱 **Restore the Ecosystem** | A Random Forest is trained live (80/20 split). Compare your prediction with the model's and the actual condition, then pick restoration strategies. | Supervised ML, model evaluation, data-driven decisions |

---
# 🎮 GAME CODE
## Step 0 – Install (only if needed)

In [ ]:
# Run ONCE if a library is missing, then restart the kernel. (Uncomment the next line.)
# %pip install numpy pandas matplotlib scikit-learn ipywidgets
print("If you get an ImportError, uncomment the %pip line above and run this cell again.")

## Step 1 – Imports, settings and the synthetic ocean dataset (Pandas + NumPy)

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, ConfusionMatrixDisplay

%matplotlib inline
plt.rcParams.update({"figure.facecolor": "#f4fbff", "axes.facecolor": "#eaf7fc",
                     "axes.edgecolor": "#0a3d6b", "axes.titleweight": "bold", "font.size": 10})

SEED = 42
GRID = 6                                   # 6 x 6 = 36 ocean regions
FEATURES = ["temperature", "oxygen", "plastic", "water_quality", "animal_pop"]
FEATURE_LABELS = {"temperature": "Temperature (°C)", "oxygen": "Oxygen (mg/L)", "plastic": "Plastic pollution",
                  "water_quality": "Water quality", "animal_pop": "Animal population"}
CLASSES = ["Healthy", "Warning", "Critical"]
THRESH = dict(oxygen=5.0, plastic=50, temperature=28.0, water_quality=55, animal_pop=70)   # "stress signals"
BOUNDS = {"temperature": (15, 36), "oxygen": (1, 10), "plastic": (0, 150), "water_quality": (5, 100), "animal_pop": (5, 250)}

REGION_NAMES = ["Sunlit Shallows", "Coral Garden", "Turtle Cove", "Kelp Forest", "Dolphin Bay", "Seagrass Meadow",
                "Reef Wall", "Anemone Fields", "Sandy Plateau", "Shipwreck Ridge", "Jellyfish Drift", "Blue Lagoon",
                "Twilight Slope", "Whale Corridor", "Octopus Den", "Oil Rig Shadow", "Harbour Mouth", "Plastic Gyre Edge",
                "Cold Current", "Manta Passage", "Sunken Canyon", "Trash Basin", "Vent Meadow", "Silent Plain",
                "Abyss Rim", "Lantern Fish Zone", "Tidal Flats", "Black Smoker Field", "Drift Net Alley", "Midnight Shelf",
                "Deep Trench", "Hadal Gate", "Lost City Rocks", "Brine Pool", "Whale Fall", "Abyssal Floor"]
ANIMALS = ["🐢 Sea turtle", "🐬 Dolphin", "🐟 Reef fish", "🦈 Shark", "🐙 Octopus", "🦀 Crab", "🐳 Whale", "🐠 Clownfish"]
POLLUTION_TYPES = {"🛍️ Plastic waste": 2, "🛢️ Oil spill": 3, "☣️ Contaminated water": 2}   # cleanup cost in units


def health_score(d):
    # Ocean-health score 0-100 computed from the five measurements.
    temp_pen = ((d["temperature"] - 26).abs() / 8).clip(0, 1)
    s = 100 * (0.25 * ((d["oxygen"] - 2) / 6).clip(0, 1) + 0.25 * (1 - (d["plastic"] / 100).clip(0, 1))
               + 0.20 * (d["water_quality"] / 100).clip(0, 1) + 0.15 * (1 - temp_pen)
               + 0.15 * (d["animal_pop"] / 150).clip(0, 1))
    return s.round(1)


def categorize(score):
    return pd.cut(score, [-np.inf, 42, 62, np.inf], labels=["Critical", "Warning", "Healthy"]).astype(str)


def refresh(d):
    # (Re)compute health score and category after any change in the data.
    d = d.copy()
    d["health_score"] = health_score(d)
    d["health"] = categorize(d["health_score"])
    return d


def make_ocean(n, seed, outlier_frac=0.0, s=None):
    # Synthetic but realistic data: a hidden 'stress' factor s drives all variables (+ random noise).
    rng = np.random.default_rng(seed)
    if s is None:
        s = rng.uniform(0, 1, n)
    d = pd.DataFrame({"temperature": 24 + 6 * s + rng.normal(0, 1.0, n),
                      "oxygen": 8.5 - 5.5 * s + rng.normal(0, 0.6, n),
                      "plastic": 10 + 60 * s + rng.normal(0, 7, n),
                      "water_quality": 95 - 60 * s + rng.normal(0, 6, n),
                      "animal_pop": 150 - 110 * s + rng.normal(0, 12, n)})
    k = int(round(outlier_frac * n))
    if k:                                                    # a few extreme pollution spikes (oil / waste dumps)
        hot = np.argsort(np.asarray(s))[-max(int(0.35 * n), k):]          # spikes appear in the most stressed regions
        idx = rng.choice(hot, k, replace=False)
        d.loc[idx, "plastic"] = rng.uniform(112, 140, k)
    for col, (lo, hi) in BOUNDS.items():
        d[col] = d[col].clip(lo, hi).round(1)
    return refresh(d)


def build_world(seed=SEED):
    # The 36-region game world: deeper regions are (on average) more stressed -> spatial bias for Mission 1.
    rng = np.random.default_rng(seed)
    rows, cols = np.repeat(np.arange(GRID), GRID), np.tile(np.arange(GRID), GRID)
    s = np.clip(0.4 * rng.uniform(0, 1, GRID * GRID) + 0.62 * rows / (GRID - 1), 0, 1)
    w = make_ocean(GRID * GRID, seed, outlier_frac=0.11, s=s)
    w.insert(0, "region_id", [f"R{i + 1:02d}" for i in range(GRID * GRID)])
    w.insert(1, "name", REGION_NAMES)
    w["row"], w["col"] = rows, cols
    w["animal"] = rng.choice(ANIMALS, GRID * GRID)
    w["pollution_type"] = rng.choice(list(POLLUTION_TYPES), GRID * GRID)
    return w


world_preview = build_world()
print("Dataset columns:", list(world_preview.columns))
world_preview.head()

## Step 2 – Game state, scoring, HUD and animated underwater visuals

In [ ]:
# ---------------------------------------------------------------- game state
def new_game():
    global G
    w = build_world()
    base = {f: (w[f].mean(), w[f].std(ddof=0)) for f in FEATURES}
    tdev = (w["temperature"] - 26).abs()
    base["tdev"] = (tdev.mean(), tdev.std(ddof=0))
    G = dict(df=w, df0=w.copy(), base=base, score=0, oxygen=100, animals=0, cleaned=0, restored=0,
             missions=set(), explored=set(), pos=(0, 0), ml=None)

def add_score(n):
    G["score"] += n

def ocean_health():
    return float(G["df"]["health_score"].mean())

def update_region(i, **changes):
    # Change measurements of region i with functions {column: f(old)->new}; health is recomputed.
    d = G["df"]
    for col, f in changes.items():
        lo, hi = BOUNDS[col]
        d.loc[i, col] = round(float(np.clip(f(d.loc[i, col]), lo, hi)), 1)
    G["df"] = refresh(d)

def risk_reasons(r):
    out = []
    if r["oxygen"] < THRESH["oxygen"]: out.append(f"oxygen is low ({r['oxygen']} mg/L)")
    if r["plastic"] > THRESH["plastic"]: out.append(f"plastic level is high ({r['plastic']:.0f})")
    if r["temperature"] > THRESH["temperature"]: out.append(f"water is too warm ({r['temperature']} °C)")
    if r["water_quality"] < THRESH["water_quality"]: out.append(f"water quality is poor ({r['water_quality']:.0f}/100)")
    if r["animal_pop"] < THRESH["animal_pop"]: out.append(f"animal population is low ({r['animal_pop']:.0f})")
    return out

def outlier_threshold(x, method="IQR method", k=1.5):
    # Dynamic pollution threshold - recomputed from whatever data currently exists.
    if method.startswith("IQR"):
        q1, q3 = x.quantile(0.25), x.quantile(0.75)
        return float(q3 + k * (q3 - q1))
    return float(x.mean() + k * x.std(ddof=0))

# ---------------------------------------------------------------- visuals
CSS = '''<style>
.oc-scene{position:relative;overflow:hidden;border-radius:16px;background:linear-gradient(180deg,#7be0f5 0%,#1fa0c9 35%,#0b4f86 75%,#062a52 100%);font-family:'Segoe UI',Arial,sans-serif;color:#fff;text-align:center;margin-bottom:8px}
.oc-scene h1{margin:0;font-size:34px;text-shadow:0 3px 8px #00294d;position:relative;z-index:3;color:#fff}
.oc-scene p{margin:2px 0;position:relative;z-index:3;font-size:15px;color:#fff}
.oc-fish{position:absolute;font-size:26px;z-index:2}
.oc-bub{position:absolute;bottom:-20px;border-radius:50%;border:2px solid rgba(255,255,255,.75);background:rgba(255,255,255,.2);z-index:1}
.oc-floor{position:absolute;bottom:0;left:0;right:0;font-size:24px;letter-spacing:6px;z-index:2}
.oc-conf{position:absolute;top:-10%;font-size:22px;z-index:2}
@keyframes oc-swim{0%{left:-8%}100%{left:104%}}
@keyframes oc-swimr{0%{left:104%}100%{left:-8%}}
@keyframes oc-rise{0%{transform:translateY(0);opacity:.9}100%{transform:translateY(-420px);opacity:0}}
@keyframes oc-glow{0%,100%{text-shadow:0 0 10px #ffd54a,0 0 20px #ff9800}50%{text-shadow:0 0 25px #fff176,0 0 45px #ffca28}}
@keyframes oc-fall{0%{top:-10%}100%{top:105%}}
</style>'''

def banner(title="🌊 OCEAN RESCUE", subtitle="THE DEEP SEA ADVENTURE", height=120, victory=False):
    rng = np.random.default_rng(5)
    bub = "".join(f"<div class='oc-bub' style='left:{int(rng.integers(2, 97))}%;width:{int(sz)}px;height:{int(sz)}px;"
                  f"animation:oc-rise {rng.uniform(4, 9):.1f}s linear {-rng.uniform(0, 8):.1f}s infinite'></div>"
                  for sz in rng.integers(6, 17, 14))
    fishes = [("🐟", 20, 11, 0, True), ("🐠", 45, 15, -5, True), ("🐢", 62, 22, -9, True),
              ("🐬", 30, 17, -3, False), ("🦈", 55, 26, -14, False), ("🐳", 8, 30, -20, True), ("🚢", 3, 40, -10, True)]
    fh = "".join(f"<div class='oc-fish' style='top:{t}%;animation:{'oc-swim' if fl else 'oc-swimr'} {dur}s linear {dl}s infinite;"
                 f"{'transform:scaleX(-1);' if fl and e in ('🐟', '🐠', '🐢', '🐬') else ''}'>{e}</div>"
                 for e, t, dur, dl, fl in fishes)
    conf = ""
    if victory:
        conf = "".join(f"<div class='oc-conf' style='left:{int(rng.integers(2, 97))}%;animation:oc-fall {rng.uniform(3, 6):.1f}s linear "
                       f"{-rng.uniform(0, 5):.1f}s infinite'>{rng.choice(['🎉', '✨', '🌟', '🐚', '🎊'])}</div>" for _ in range(22))
    glow = "animation:oc-glow 1.6s ease-in-out infinite;" if victory else ""
    floor = "🪸🌿🪸🌱🌿🪸🐚🌿🪸🌱🪸🌿🪸🌱🌿🪸"
    return (CSS + f"<div class='oc-scene' style='height:{height}px'>{bub}{fh}{conf}"
            f"<div style='padding-top:{height * 0.16:.0f}px'><h1 style='{glow}'>{title}</h1><p>{subtitle}</p></div>"
            f"<div class='oc-floor'>{floor}</div></div>")

hud_w = widgets.HTML()

def _bar(label, pct, color):
    pct = max(0, min(100, pct))
    return (f"<div style='flex:1;min-width:160px'><div style='font-size:12px;margin-bottom:2px'>{label}</div>"
            f"<div style='background:#ffffff33;border-radius:8px;height:14px'>"
            f"<div style='width:{pct:.0f}%;background:{color};height:14px;border-radius:8px'></div></div></div>")

def refresh_hud():
    hp, ox = ocean_health(), G["oxygen"]
    oxc = "#4fe3a1" if ox > 50 else "#ffc43d" if ox > 25 else "#ff5d5d"
    hud_w.value = ("<div style='display:flex;gap:16px;align-items:center;flex-wrap:wrap;background:#083c6b;color:#fff;"
                   "padding:8px 14px;border-radius:12px;font-family:Segoe UI,Arial'>"
                   f"<div style='font-size:18px'>⭐ <b>{G['score']}</b></div>"
                   + _bar(f"🫧 Oxygen {ox}%", ox, oxc)
                   + _bar(f"🎯 Missions {len(G['missions'])}/4", len(G["missions"]) / 4 * 100, "#4dc3ff")
                   + _bar(f"🌊 Ocean health {hp:.0f}%", hp, "#7CFC9A") + "</div>")

def note(html, kind="info"):
    col = {"info": ("#e8f6ff", "#1fa0c9"), "good": ("#e6f9ee", "#1b9e5a"), "bad": ("#fdeaea", "#d64545"),
           "concept": ("#fff6dc", "#f5a623")}[kind]
    return widgets.HTML(f"<div style='background:{col[0]};border-left:6px solid {col[1]};padding:9px 14px;border-radius:8px;"
                        f"font-size:14px;color:#12324d;font-family:Segoe UI,Arial;margin:4px 0'>{html}</div>")

def concept(title, text):
    return note(f"<b>🧠 Data Science concept discovered: {title}</b><br>{text}", "concept")

def btn(label, cb, style="primary", width="230px"):
    b = widgets.Button(description=label, button_style=style, layout=widgets.Layout(width=width, height="42px"))
    b.on_click(cb)
    return b

screen = widgets.Output()          # the single 'screen' that every game page is drawn into

def head(title, sub=""):
    clear_output(wait=True)
    refresh_hud()
    display(widgets.HTML(banner(title, sub, height=110)))
    display(hud_w)

def stat_chips(r):
    items = [("🌡️", f"{r['temperature']:.1f} °C"), ("💧 O₂", f"{r['oxygen']:.1f} mg/L"), ("🛍️ Plastic", f"{r['plastic']:.0f}"),
             ("🧪 Quality", f"{r['water_quality']:.0f}/100"), ("🐾 Population", f"{r['animal_pop']:.0f}")]
    return "".join(f"<span style='display:inline-block;background:#e3f4fb;color:#12324d;border-radius:8px;padding:3px 8px;"
                   f"margin:2px;font-size:13px'>{a} <b>{b}</b></span>" for a, b in items)

def finish_mission(n, title, text):
    G["missions"].add(n)
    refresh_hud()
    display(concept(title, text))
    display(btn("🗺️ Back to Ocean Map", lambda b: screen_hub(), "success", "240px"))

new_game()
refresh_hud()
print("Game engine ready ✅")

## Step 3 – Main menu, instructions, learning objectives and the interactive ocean map

In [ ]:
OBJECTIVES = ["Understand data exploration and sampling.", "Understand classification using environmental data.",
              "Interpret correlation between environmental variables.", "Perform outlier detection and data visualization.",
              "Understand machine learning classification.", "Make decisions using data analysis."]

def screen_menu():
    with screen:
        clear_output(wait=True)
        display(widgets.HTML(banner("🌊 OCEAN RESCUE 🌊", "THE DEEP SEA ADVENTURE – restore the ocean with Data Science!", height=250)))
        display(widgets.VBox([btn("▶  Start Game", lambda b: start_game(), "success", "300px"),
                              btn("📖  Instructions", lambda b: screen_instructions(), "info", "300px"),
                              btn("🚪  Exit", lambda b: screen_exit(), "danger", "300px")],
                             layout=widgets.Layout(align_items="center", gap="10px")))

def start_game():
    new_game()
    screen_objectives()

def screen_instructions():
    with screen:
        head("📖 Instructions", "How to play")
        display(note("<b>Goal:</b> complete 4 missions and push the <b>Ocean Health %</b> as high as possible.<br>"
                     "🧭 <b>Mission 1</b> – steer the submarine with the arrow buttons (each move costs oxygen), discover regions, then pick a sample.<br>"
                     "🐢 <b>Mission 2</b> – read the water data and decide which animals need rescue <i>now</i>.<br>"
                     "🧹 <b>Mission 3</b> – find unusual pollution with the IQR / Z-score tools, then spend limited cleanup units wisely.<br>"
                     "🌱 <b>Mission 4</b> – a machine-learning model predicts ocean health; choose the best restoration strategies.<br><br>"
                     "<b>Scoring:</b> new region +10 · endangered area identified +15 · animal rescued +20 · polluted region cleaned +20 · restoration decision +30."))
        display(btn("⬅️ Back to menu", lambda b: screen_menu(), "primary"))

def screen_exit():
    with screen:
        clear_output(wait=True)
        display(widgets.HTML(banner("👋 Thanks for playing!", "The ocean needs data-driven heroes like you.", height=160)))
        display(btn("🔄 Back to menu", lambda b: screen_menu(), "primary"))

def screen_objectives():
    with screen:
        head("🎯 Your Mission", "Learning objectives")
        display(note("🌍 <b>The story:</b> pollution, abnormal temperatures and falling animal numbers are putting the ocean in danger. "
                     "You are an underwater explorer – use <b>data</b> to explore, rescue, clean and restore."))
        display(note("<b>After this game you will be able to:</b><br>" + "<br>".join(f"{i + 1}. {o}" for i, o in enumerate(OBJECTIVES))))
        display(btn("🗺️ Open the Ocean Map", lambda b: screen_hub(), "success", "260px"))

MISSION_INFO = {1: ("🧭", "Explore the Ocean", "Sampling & data exploration", (12, 20)),
                2: ("🐢", "Save the Marine Animals", "Classification & correlation", (60, 28)),
                3: ("🧹", "Pollution Detection & Cleanup", "Outliers & visualisation", (35, 62)),
                4: ("🌱", "Restore the Ecosystem", "Machine learning", (72, 68))}

def screen_hub():
    runners = {1: screen_m1, 2: screen_m2, 3: screen_m3, 4: screen_m4}
    done = G["missions"]
    with screen:
        head("🗺️ Ocean Map", "Choose your next destination")
        pins = ""
        for k, (ico, nm, con, (x, y)) in MISSION_INFO.items():
            unlocked = k == 1 or (k - 1) in done
            state = "✅" if k in done else ("📍" if unlocked else "🔒")
            pins += (f"<div style='position:absolute;left:{x}%;top:{y}%;transform:translate(-50%,-50%);text-align:center;"
                     f"opacity:{1 if unlocked else .45}'><div style='font-size:44px'>{ico}</div>"
                     f"<div style='background:#ffffffcc;color:#12324d;border-radius:8px;padding:2px 8px;font-size:12px;font-weight:bold'>"
                     f"{state} Mission {k}</div></div>")
        display(widgets.HTML("<div style='position:relative;height:260px;border-radius:16px;font-family:Segoe UI,Arial;"
                             "background:linear-gradient(180deg,#5fd3ee,#1d8fc4 40%,#0a3d6b 80%,#041e3c);overflow:hidden'>"
                             "<div style='position:absolute;left:4%;bottom:4px;font-size:28px'>🪸🌿🪸🐚🌿🪸🌱🪸🌿🪸🐚🌿🪸🌱</div>"
                             "<div style='position:absolute;left:46%;top:8%;font-size:34px'>🚢</div>" + pins + "</div>"))
        for k, (ico, nm, con, _) in MISSION_INFO.items():
            unlocked = k == 1 or (k - 1) in done
            b = btn(f"{ico} Mission {k}: {nm}" + ("  ✅" if k in done else ""), (lambda b, k=k: runners[k]()),
                    "success" if k in done else "primary", "380px")
            b.disabled = not unlocked
            display(widgets.HBox([b, widgets.HTML(f"<span style='color:#444;font-size:13px'>&nbsp;Concept: {con}</span>")]))
        if len(done) == 4:
            display(btn("🏆 See Final Report", lambda b: screen_final(), "warning", "380px"))
        display(btn("⬅️ Main menu", lambda b: screen_menu(), "", "160px"))

## Step 4 – Mission 1: Explore the Ocean (sampling & data exploration)

In [ ]:
DEPTH_COLORS = ["#4fd1e8", "#2fb5d8", "#1f8fc0", "#17699f", "#0f4a7c", "#082f5c"]
MOVE_COST = 4

def discover(r, c):
    if (r, c) in G["explored"]:
        return False
    G["explored"].add((r, c))
    add_score(10)
    return True

def screen_m1():
    G["oxygen"], G["pos"], G["explored"] = 100, (0, 0), set()
    discover(0, 0)
    map_w, status_w = widgets.HTML(), widgets.HTML()
    finish_b = btn("🔬 Finish exploring → choose samples", lambda b: screen_m1_sample(), "success", "340px")

    def draw(msg=""):
        d = G["df"]
        rows = ""
        for r in range(GRID):
            rows += "<tr>"
            for c in range(GRID):
                x = d.iloc[r * GRID + c]
                here = (r, c) == G["pos"]
                border = "3px solid #ffd54a" if here else "1px solid #ffffff55"
                if (r, c) in G["explored"]:
                    dot = "🟢" if x["plastic"] < 35 else "🟡" if x["plastic"] < 60 else "🔴"
                    body = (f"<div style='font-size:26px'>{'🤿' if here else x['animal'].split()[0]}</div>"
                            f"<div style='font-size:10px'>{x['region_id']} · {x['temperature']:.0f}°C {dot}</div>")
                else:
                    body = "<div style='font-size:26px'>❓</div><div style='font-size:10px'>unexplored</div>"
                rows += (f"<td style='width:98px;height:70px;text-align:center;background:{DEPTH_COLORS[r]};color:#fff;"
                         f"border:{border};border-radius:8px;opacity:{1 if (r, c) in G['explored'] else .72}'>{body}</td>")
            rows += "</tr>"
        map_w.value = ("<div style='font-family:Segoe UI,Arial;color:#fff'><div style='color:#12324d;font-size:12px;margin-bottom:3px'>"
                       "Surface ⬆ … depth ⬇ &nbsp;|&nbsp; 🟢 clean · 🟡 moderate · 🔴 heavy plastic</div>"
                       f"<table style='border-spacing:3px;background:#06284a;padding:6px;border-radius:12px'>{rows}</table></div>")
        ex = len(G["explored"])
        status_w.value = (f"<div style='font-size:14px;color:#12324d'>{msg}<br>🧭 Explored <b>{ex}/{GRID * GRID}</b> regions "
                          f"(explore at least <b>8</b>) · 🫧 oxygen <b>{G['oxygen']}%</b> · each move costs {MOVE_COST}%</div>")
        finish_b.disabled = ex < 8
        for b in pad:
            b.disabled = G["oxygen"] < MOVE_COST
        refresh_hud()

    def move(dr, dc):
        r, c = G["pos"]
        nr, nc = r + dr, c + dc
        if not (0 <= nr < GRID and 0 <= nc < GRID):
            draw("🌊 You hit the edge of the map."); return
        G["pos"] = (nr, nc)
        G["oxygen"] = max(0, G["oxygen"] - MOVE_COST)
        x = G["df"].iloc[nr * GRID + nc]
        if discover(nr, nc):
            msg = (f"✨ <b>New region discovered: {x['name']}</b> (+10 points) – {x['animal']}, {x['temperature']:.1f} °C, "
                   f"O₂ {x['oxygen']:.1f} mg/L, plastic {x['plastic']:.0f}.")
        else:
            msg = f"You are at {x['name']} (already explored)."
        if G["oxygen"] < MOVE_COST:
            msg += " 🫧 <b>Out of oxygen – time to surface and choose your samples!</b>"
        draw(msg)

    def mk(label, dr, dc):
        b = widgets.Button(description=label, layout=widgets.Layout(width="70px", height="44px"), button_style="info")
        b.on_click(lambda _: move(dr, dc)); return b
    up, left, down, right = mk("⬆️", -1, 0), mk("⬅️", 0, -1), mk("⬇️", 1, 0), mk("➡️", 0, 1)
    pad = [up, left, down, right]
    dpad = widgets.GridBox([widgets.Label(""), up, widgets.Label(""), left, down, right],
                           layout=widgets.Layout(grid_template_columns="repeat(3,72px)", grid_gap="4px"))
    with screen:
        head("🧭 Mission 1: Explore the Ocean", "Explore regions and observe the ecosystem")
        display(note("🎯 <b>Objective:</b> drive the submarine, discover regions (+10 each) and observe animals, temperature and pollution. "
                     "Oxygen is limited – <b>choose where to go wisely</b>; you will later pick a <i>sample</i> from what you explored."))
        draw("Use the arrow buttons to move.")
        display(widgets.HBox([map_w, widgets.VBox([widgets.HTML("<b style='color:#12324d'>🚤 Controls</b>"), dpad])],
                             layout=widgets.Layout(gap="20px", align_items="center")))
        display(status_w)
        display(finish_b)

def screen_m1_sample():
    d = G["df"]
    ids = sorted(r * GRID + c for r, c in G["explored"])
    opts = [(f"{d.iloc[i]['region_id']} {d.iloc[i]['name']} · {d.iloc[i]['animal'].split()[0]} · {d.iloc[i]['temperature']:.1f}°C · plastic {d.iloc[i]['plastic']:.0f}", i) for i in ids]
    sel = widgets.SelectMultiple(options=opts, rows=min(12, len(opts)), layout=widgets.Layout(width="720px"))
    out = widgets.Output()
    go = btn("📊 Analyse my sample", lambda b: analyse(), "primary", "240px")

    def analyse():
        idx = list(sel.value)
        with out:
            clear_output(wait=True)
            if not 5 <= len(idx) <= 12:
                display(note("⚠️ Please select between <b>5 and 12</b> regions (hold Ctrl / Shift to select several).", "bad")); return
            go.disabled = True; sel.disabled = True
            pop, samp = d[FEATURES], d.iloc[idx][FEATURES]
            diff = ((samp.mean() - pop.mean()) / pop.std(ddof=0))
            cmp_ = pd.DataFrame({"Whole ocean (population)": pop.mean(), "Your sample": samp.mean(), "Std. difference": diff}).round(2)
            representative = bool((diff[["temperature", "oxygen", "plastic"]].abs() < 0.25).all())
            rand_means = np.array([d["plastic"].sample(len(idx), random_state=k).mean() for k in range(400)])
            fig, ax = plt.subplots(1, 2, figsize=(13, 4))
            ax[0].hist(rand_means, bins=25, color="#5fc8e8", edgecolor="white", label="400 random samples")
            ax[0].axvline(pop["plastic"].mean(), color="k", ls="--", label="True ocean mean")
            ax[0].axvline(samp["plastic"].mean(), color="#e4572e", lw=3, label="YOUR sample mean")
            ax[0].set_title("Average plastic level: your sample vs random samples"); ax[0].set_xlabel("Sample mean plastic level"); ax[0].legend()
            ax[1].scatter(d["col"], d["row"], s=130, c="#b8dbe8", label="not sampled")
            ax[1].scatter(d.iloc[idx]["col"], d.iloc[idx]["row"], s=150, c="#e4572e", label="your sample")
            ax[1].invert_yaxis(); ax[1].set_title("Where your sample came from (top = surface)"); ax[1].legend(loc="lower right")
            ax[1].set_xticks(range(GRID)); ax[1].set_yticks(range(GRID)); plt.tight_layout(); plt.show()
            display(cmp_)
            if representative:
                add_score(20)
                display(note("✅ <b>Representative sample! +20 points.</b> Your sample means are close to the whole ocean (all differences &lt; 0.25 standard deviations).", "good"))
            else:
                worst = diff[["temperature", "oxygen", "plastic"]].abs().idxmax()
                display(note(f"⚠️ <b>Your sample is biased.</b> It differs most on <b>{FEATURE_LABELS[worst]}</b> "
                             f"({diff[worst]:+.2f} std). Samples taken from one area (e.g. only shallow water) give a distorted picture of the whole ocean.", "bad"))
            finish_mission(1, "Sampling & data exploration",
                           "A <b>sample</b> is a small part of a larger <b>population</b> (all 36 regions). A <b>representative</b> sample spreads over the whole population, "
                           "so its averages match the population's. A <b>biased</b> sample (e.g. only shallow, clean water) leads to wrong conclusions.")
    with screen:
        head("🔬 Mission 1: Choose your sample", "Which regions will you investigate in detail?")
        display(note("You can only investigate a few regions with your lab equipment. <b>Select 5–12 explored regions</b> that you think "
                     "best represent the <i>whole</i> ocean. Then compare your sample with the full dataset."))
        display(sel, go, out)

## Step 5 – Mission 2: Save the Marine Animals (classification, features, correlation)

In [ ]:
def screen_m2():
    G["oxygen"] = 100
    d = G["df"]
    rng = np.random.default_rng(11)
    pick = lambda cat, k: [int(i) for i in rng.choice(d.index[d["health"] == cat], min(k, int((d["health"] == cat).sum())), replace=False)]
    cases = pick("Critical", 4) + pick("Warning", 2) + pick("Healthy", 2)
    rest = [int(i) for i in d.index if int(i) not in cases]
    while len(cases) < 8:
        cases.append(rest.pop(int(rng.integers(len(rest)))))
    rng.shuffle(cases)
    toggles, out = {}, widgets.Output()
    rows = []
    for i in cases:
        r = d.loc[i]
        card = widgets.HTML(f"<div style='width:600px;color:#12324d;font-family:Segoe UI,Arial'><b>{r['animal']}</b> in <b>{r['name']}</b> ({r['region_id']})<br>{stat_chips(r)}</div>")
        t = widgets.ToggleButtons(options=["🆘 Rescue now", "✅ Not urgent"], value=None, layout=widgets.Layout(width="330px"))
        toggles[i] = t
        rows.append(widgets.HBox([card, t], layout=widgets.Layout(border="1px solid #9ad", padding="4px", margin="3px 0", align_items="center")))
    go = btn("📨 Submit rescue decisions", lambda b: submit(), "primary", "280px")

    def submit():
        if any(t.value is None for t in toggles.values()):
            with out:
                clear_output(wait=True); display(note("⚠️ Please decide for <b>every</b> animal first.", "bad"))
            return
        go.disabled = True
        for t in toggles.values(): t.disabled = True
        msgs, rescued = [], 0
        for i, t in toggles.items():
            r = G["df"].loc[i]; danger = r["health"] == "Critical"; why = "; ".join(risk_reasons(r)) or "no major stress signals"
            rescue = t.value.startswith("🆘")
            if rescue and danger and G["oxygen"] >= 10:
                G["oxygen"] -= 10; add_score(15 + 20); G["animals"] += 1; rescued += 1
                update_region(i, oxygen=lambda x: x + 1.5, plastic=lambda x: x - 10, animal_pop=lambda x: x + 20)
                msgs.append(("good", f"✅ <b>{r['animal']} ({r['name']}) rescued!</b> +15 (danger identified) +20 (rescue). Signals: {why}."))
            elif rescue and danger:
                msgs.append(("bad", f"🫧 {r['animal']} needed rescue but you ran out of oxygen. Signals: {why}."))
            elif rescue:
                G["oxygen"] -= 10
                msgs.append(("bad", f"❌ <b>{r['name']}</b> was classified <b>{r['health']}</b> – it was not in danger. You wasted 10% oxygen. Signals: {why}."))
            elif danger:
                update_region(i, animal_pop=lambda x: x * 0.85, plastic=lambda x: x + 5)
                msgs.append(("bad", f"⚠️ <b>Missed!</b> {r['animal']} in {r['name']} was in <b>Critical</b> condition ({why}). Its population dropped by 15%, hurting ocean health."))
            else:
                add_score(5)
                msgs.append(("good", f"👍 {r['name']} is <b>{r['health']}</b> – waiting was fine (+5). Signals: {why}."))
        with out:
            clear_output(wait=True); refresh_hud()
            for k, m in msgs: display(note(m, k))
            display(note(f"🐾 Animals rescued this mission: <b>{rescued}</b> · Ocean health is now <b>{ocean_health():.1f}%</b>.", "info"))
            analysis()

    def analysis():
        d2 = G["df"]
        display(HTML("<h3 style='color:#0a3d6b'>🔎 What does the data say? (correlation &amp; feature identification)</h3>"))
        corr = d2[FEATURES].corr()
        tree = DecisionTreeClassifier(max_depth=3, random_state=SEED).fit(d2[FEATURES], d2["health"])
        imp = pd.Series(tree.feature_importances_, FEATURES).sort_values()
        fig, ax = plt.subplots(1, 2, figsize=(13, 4.2))
        im = ax[0].imshow(corr, cmap="coolwarm", vmin=-1, vmax=1)
        ax[0].set_xticks(range(5)); ax[0].set_yticks(range(5))
        ax[0].set_xticklabels(FEATURES, rotation=35, ha="right"); ax[0].set_yticklabels(FEATURES)
        for a in range(5):
            for b in range(5): ax[0].text(b, a, f"{corr.iloc[a, b]:.2f}", ha="center", va="center", fontsize=9)
        ax[0].set_title("Correlation matrix"); plt.colorbar(im, ax=ax[0], fraction=0.046)
        ax[1].barh([FEATURE_LABELS[k] for k in imp.index], imp.values, color="#2a9fd6")
        ax[1].set_title("Which features best separate Healthy / Warning / Critical?\n(Decision Tree feature importance)")
        plt.tight_layout(); plt.show()
        pairs = corr.where(~np.eye(5, dtype=bool)).abs().stack().sort_values(ascending=False)
        (a, b) = pairs.index[0]
        display(note(f"📈 Strongest relationship: <b>{a}</b> vs <b>{b}</b> (r = {corr.loc[a, b]:+.2f}). "
                     f"Most important feature for classification: <b>{FEATURE_LABELS[imp.index[-1]]}</b>.", "info"))
        q = widgets.RadioButtons(options=["Plastic pollution definitely CAUSES the oxygen loss, because the correlation is so strong.",
                                          "They move together, but correlation alone cannot prove cause – a hidden factor (overall water stress) may drive both.",
                                          "There is no relationship, because the points do not lie exactly on a line."],
                                 value=None, layout=widgets.Layout(width="900px"))
        res = widgets.Output()
        def check(_):
            with res:
                clear_output(wait=True)
                if q.value is None: return
                chk.disabled = True; q.disabled = True
                if q.value.startswith("They move"):
                    add_score(10); refresh_hud(); display(note("✅ <b>Correct (+10).</b> Correlation measures how variables move together – it does not prove one causes the other.", "good"))
                else:
                    display(note("❌ Not quite. A strong correlation means the variables move together, but cause needs more evidence (experiments, a hidden common factor…). The right answer is the 2nd statement.", "bad"))
                finish_mission(2, "Classification, feature identification & correlation",
                               "<b>Classification</b> puts each region in a category (Healthy / Warning / Critical) using <b>features</b> such as oxygen and plastic. "
                               "<b>Correlation</b> (r from −1 to +1) shows how two variables move together – <b>correlation ≠ causation</b>.")
        chk = btn("Check my answer", check, "primary", "200px")
        display(HTML("<b style='color:#12324d'>🤔 Which statement is correct?</b>"), q, chk, res)
    with screen:
        head("🐢 Mission 2: Save the Marine Animals", "Read the data – who needs help right now?")
        display(note("🎯 <b>Objective:</b> for each animal decide <b>Rescue now</b> or <b>Not urgent</b>. A rescue costs 10% oxygen, so do not waste it!<br>"
                     f"<b>Stress signals (rule of thumb):</b> O₂ &lt; {THRESH['oxygen']} · plastic &gt; {THRESH['plastic']} · temperature &gt; {THRESH['temperature']} °C · "
                     f"quality &lt; {THRESH['water_quality']} · population &lt; {THRESH['animal_pop']}. Regions with many signals are usually <b>Critical</b>."))
        for r in rows: display(r)
        display(go, out)

## Step 6 – Mission 3: Pollution Detection & Ocean Cleanup (outliers, statistics, visualisation)

In [ ]:
CLEAN_BUDGET = 12

def screen_m3():
    G["oxygen"] = 100
    before = G["df"].copy()
    health_before = ocean_health()
    cand = before.sort_values("plastic", ascending=False).head(14).index.tolist()
    method = widgets.Dropdown(options=["IQR method", "Z-score method"], value="IQR method", description="Method:")
    kslider = widgets.FloatSlider(value=1.5, min=0.5, max=3.5, step=0.1, description="k:", readout_format=".1f")
    info, budget_w, plot_out, res_out = widgets.HTML(), widgets.HTML(), widgets.Output(), widgets.Output()
    toggles = {}

    def update(_=None):
        if method.value.startswith("Z") and abs(kslider.value - 1.5) < 1e-9: kslider.value = 2.0; return
        thr = outlier_threshold(before["plastic"], method.value, kslider.value)
        flagged = before[before["plastic"] > thr]
        formula = "Q3 + k × IQR" if method.value.startswith("IQR") else "mean + k × std-dev"
        info.value = (f"<div style='color:#12324d;font-size:14px'>📏 Dynamic threshold ({formula}) = <b>{thr:.1f}</b> · "
                      f"<b>{len(flagged)}</b> unusual regions flagged: {', '.join(flagged['region_id']) or 'none'}</div>")
        with plot_out:
            clear_output(wait=True)
            fig, ax = plt.subplots(figsize=(13, 3.2))
            ax.bar(before["region_id"], before["plastic"], color=["#e4572e" if v > thr else "#2a9fd6" for v in before["plastic"]])
            ax.axhline(thr, color="k", ls="--", label=f"threshold = {thr:.1f}")
            ax.set_title("Plastic pollution by region (red = unusual)"); ax.set_ylabel("Plastic level")
            ax.tick_params(axis="x", rotation=90, labelsize=7); ax.legend(); plt.tight_layout(); plt.show()

    def used():
        return sum(POLLUTION_TYPES[before.loc[i, "pollution_type"]] for i, t in toggles.items() if t.value)

    def on_toggle(change):
        u = used()
        if u > CLEAN_BUDGET:
            change["owner"].value = False
            budget_w.value = "<div style='color:#d64545'>⚠️ Not enough cleanup units for that region!</div>"; return
        budget_w.value = f"<div style='color:#12324d;font-size:14px'>🔧 Cleanup units used: <b>{u}/{CLEAN_BUDGET}</b></div>"

    for i in cand:
        r = before.loc[i]; cost = POLLUTION_TYPES[r["pollution_type"]]
        t = widgets.ToggleButton(description=f"{r['region_id']} {r['pollution_type'].split()[0]} plastic {r['plastic']:.0f} · cost {cost}",
                                 layout=widgets.Layout(width="300px", height="38px"), tooltip=f"{r['name']} – {r['pollution_type']}")
        t.observe(on_toggle, names="value"); toggles[i] = t
    budget_w.value = f"<div style='color:#12324d;font-size:14px'>🔧 Cleanup units used: <b>0/{CLEAN_BUDGET}</b></div>"
    go = btn("🧹 Start cleanup!", lambda b: clean(), "success", "240px")

    def clean():
        chosen = [i for i, t in toggles.items() if t.value]
        if not chosen:
            with res_out:
                clear_output(wait=True); display(note("⚠️ Select at least one region to clean.", "bad"))
            return
        go.disabled = True
        for t in toggles.values(): t.disabled = True
        q3 = before["plastic"].quantile(0.75); fence = outlier_threshold(before["plastic"], "IQR method", 1.5)
        msgs, ok = [], 0
        for i in chosen:
            r = before.loc[i]
            if r["plastic"] >= q3:
                add_score(20); ok += 1; G["cleaned"] += 1
                extra = ""
                if r["plastic"] > fence:
                    add_score(10); extra = f" 🎯 It was a <b>statistical outlier</b> (above the IQR fence {fence:.1f}) – priority target! +10 bonus."
                msgs.append(("good", f"✅ <b>{r['name']}</b> ({r['pollution_type']}) cleaned: plastic {r['plastic']:.0f} → {r['plastic'] * 0.3:.0f}. +20.{extra}"))
            else:
                msgs.append(("bad", f"❌ <b>{r['name']}</b> was only lightly polluted ({r['plastic']:.0f}, below Q3 = {q3:.0f}). You spent {POLLUTION_TYPES[r['pollution_type']]} units without much benefit."))
            update_region(i, plastic=lambda x: x * 0.3, water_quality=lambda x: x + 15, oxygen=lambda x: x + 1.0)
        after = G["df"]
        thr_after = outlier_threshold(after["plastic"], "IQR method", 1.5)
        with res_out:
            clear_output(wait=True); refresh_hud()
            for k, m in msgs: display(note(m, k))
            ymax = max(before["plastic"].max(), after["plastic"].max()) * 1.1
            fig, ax = plt.subplots(1, 3, figsize=(16, 4), gridspec_kw={"width_ratios": [3, 3, 1.2]})
            for a, dd, ttl, th in [(ax[0], before, "BEFORE cleanup", fence), (ax[1], after, "AFTER cleanup", thr_after)]:
                a.bar(dd["region_id"], dd["plastic"], color=["#e4572e" if v > th else "#2a9fd6" for v in dd["plastic"]])
                a.axhline(th, color="k", ls="--"); a.set_ylim(0, ymax); a.set_title(f"{ttl} (threshold {th:.1f})")
                a.tick_params(axis="x", rotation=90, labelsize=7)
            ax[0].set_ylabel("Plastic level")
            ax[2].boxplot([before["plastic"], after["plastic"]]); ax[2].set_xticklabels(["Before", "After"]); ax[2].set_title("Distribution")
            plt.tight_layout(); plt.show()
            display(note(f"🌊 Ocean health: <b>{health_before:.1f}% → {ocean_health():.1f}%</b> · mean plastic {before['plastic'].mean():.1f} → {after['plastic'].mean():.1f} · "
                         f"polluted regions cleaned: <b>{ok}/{len(chosen)}</b> · unused units: {CLEAN_BUDGET - used()}. The threshold was <b>recalculated</b> from the new data "
                         f"({fence:.1f} → {thr_after:.1f}).", "info"))
            finish_mission(3, "Outlier detection, visualisation & statistical interpretation",
                           "An <b>outlier</b> is a value unusually far from the rest. The <b>IQR rule</b> flags values above Q3 + 1.5×IQR; the <b>Z-score</b> flags values more than k "
                           "standard deviations above the mean. Graphs make unusual values obvious, so limited resources go to where they matter most.")
    method.observe(update, names="value"); kslider.observe(update, names="value")
    with screen:
        head("🧹 Mission 3: Pollution Detection & Cleanup", "Find unusual pollution and clean it with limited resources")
        display(note(f"🎯 <b>Objective:</b> detect the most polluted regions, then spend <b>{CLEAN_BUDGET} cleanup units</b> (plastic 🛍️ = 2, oil 🛢️ = 3, contaminated water ☣️ = 2). "
                     "Experiment with the method and <b>k</b> to see how the dynamic threshold changes which regions are flagged. "
                     "Cleaning a region above Q3 earns +20 (+10 more for a true IQR outlier); cleaning a lightly polluted region wastes units."))
        display(widgets.HBox([method, kslider]), info, plot_out)
        update()
        display(widgets.HTML("<b style='color:#12324d'>The 14 most polluted regions – choose where to clean:</b>"))
        display(widgets.GridBox(list(toggles.values()), layout=widgets.Layout(grid_template_columns="repeat(3,310px)", grid_gap="4px")))
        display(budget_w, go, res_out)

## Step 7 – Mission 4: Restore the Ocean Ecosystem (machine learning with Scikit-learn)

In [ ]:
STRATEGIES = {"♻️ Remove plastic & oil waste": "plastic", "🌿 Oxygenate & plant seagrass": "oxygen",
              "🧊 Coral shading & cooling currents": "temperature", "🚰 Wastewater treatment": "water_quality",
              "🐢 Protected breeding zone": "animal_pop"}
EFFECTS = {"plastic": dict(plastic=lambda x: x * 0.35, water_quality=lambda x: x + 10),
           "oxygen": dict(oxygen=lambda x: x + 2.5, animal_pop=lambda x: x + 15),
           "temperature": dict(temperature=lambda x: 26 + (x - 26) * 0.3, oxygen=lambda x: x + 0.5),
           "water_quality": dict(water_quality=lambda x: x + 30, plastic=lambda x: x * 0.85),
           "animal_pop": dict(animal_pop=lambda x: x + 50)}
RANK = {"Critical": 0, "Warning": 1, "Healthy": 2}
MAX_ACTIONS = 6

def train_model():
    # Train a Random Forest on a freshly generated synthetic dataset (80/20 train-test split).
    data = make_ocean(900, seed=99, outlier_frac=0.02)
    X, y = data[FEATURES], data["health"]
    Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.2, random_state=SEED, stratify=y)
    model = RandomForestClassifier(n_estimators=150, max_depth=6, random_state=SEED).fit(Xtr, ytr)
    pred = model.predict(Xte)
    return dict(model=model, acc=float(accuracy_score(yte, pred)), cm=confusion_matrix(yte, pred, labels=CLASSES),
                imp=pd.Series(model.feature_importances_, FEATURES), n_train=len(Xtr), n_test=len(Xte))

def stress_z(r):
    # How stressed is each variable compared with the original ocean? (higher z = worse)
    b = G["base"]; z = lambda v, k: (v - b[k][0]) / (b[k][1] or 1)
    return pd.Series({"plastic": z(r["plastic"], "plastic"), "oxygen": -z(r["oxygen"], "oxygen"),
                      "temperature": z(abs(r["temperature"] - 26), "tdev"), "water_quality": -z(r["water_quality"], "water_quality"),
                      "animal_pop": -z(r["animal_pop"], "animal_pop")})

def stress_html(r):
    z, cells = stress_z(r), ""
    vals = {"plastic": f"{r['plastic']:.0f}", "oxygen": f"{r['oxygen']:.1f}", "temperature": f"{r['temperature']:.1f}°",
            "water_quality": f"{r['water_quality']:.0f}", "animal_pop": f"{r['animal_pop']:.0f}"}
    icons = {"plastic": "🛍️", "oxygen": "💧", "temperature": "🌡️", "water_quality": "🧪", "animal_pop": "🐾"}
    for k in z.index:
        hue = int(max(0, min(120, 70 - z[k] * 40)))
        cells += (f"<span style='display:inline-block;background:hsl({hue},70%,82%);color:#12324d;border-radius:8px;padding:3px 8px;"
                  f"margin:2px;font-size:13px'>{icons[k]} {vals[k]}</span>")
    return (f"<div style='width:520px;color:#12324d;font-family:Segoe UI,Arial'><b>{r['animal']} · {r['name']}</b> ({r['region_id']})<br>{cells}"
            f"<div style='font-size:11px;color:#456'>colour = stress vs. the original ocean (green = fine, red = severe)</div></div>")

def screen_m4():
    G["oxygen"] = 100
    if G["ml"] is None:
        G["ml"] = train_model()
    ml = G["ml"]; model = ml["model"]
    d0, d = G["df0"], G["df"]
    affected = d[d["health"] != "Healthy"].sort_values("health_score").head(8).index.tolist()
    out, rows = widgets.Output(), []
    pred_opts, strat_opts = ["— my prediction —"] + CLASSES, ["— skip (save action) —"] + list(STRATEGIES)
    ctrl = {}
    for i in affected:
        r = d.loc[i]
        p = widgets.Dropdown(options=pred_opts, layout=widgets.Layout(width="190px"))
        s = widgets.Dropdown(options=strat_opts, layout=widgets.Layout(width="290px"))
        ctrl[i] = (p, s)
        rows.append(widgets.HBox([widgets.HTML(stress_html(r)), widgets.VBox([p, s])],
                                 layout=widgets.Layout(border="1px solid #9ad", padding="4px", margin="3px 0", align_items="center")))
    go = btn("🌱 Apply restoration plan", lambda b: apply_plan(), "success", "300px")

    def apply_plan():
        chosen = [i for i, (p, s) in ctrl.items() if s.value in STRATEGIES]
        with out:
            clear_output(wait=True)
            if len(chosen) > MAX_ACTIONS:
                display(note(f"⚠️ You only have <b>{MAX_ACTIONS}</b> restoration actions – you selected {len(chosen)}.", "bad")); return
            go.disabled = True
            for p, s in ctrl.values(): p.disabled = s.disabled = True
            table, pts_pred, pts_ml, n_pred = [], 0, 0, 0
            for i in affected:
                p, s = ctrl[i]; r = G["df"].loc[i]
                actual = r["health"]; ml_pred = model.predict(pd.DataFrame([r[FEATURES].astype(float)]))[0]
                verdict, gain, cat_after, hs_before = "—", 0, actual, r["health_score"]
                if p.value in CLASSES:
                    n_pred += 1
                    if p.value == actual: add_score(10); gain += 10
                    pts_pred += p.value == actual
                pts_ml += ml_pred == actual
                if s.value in STRATEGIES:
                    z = stress_z(r).sort_values(ascending=False); target = STRATEGIES[s.value]
                    if target == z.index[0]: pts, w, verdict = 30, 1.0, "✅ best strategy (+30)"
                    elif target == z.index[1]: pts, w, verdict = 15, 1.0, "👍 acceptable (+15)"
                    else: pts, w, verdict = 0, 0.4, f"❌ weak fit (main stressor: {z.index[0]})"
                    add_score(pts); gain += pts
                    update_region(i, **{c: (lambda x, f=f, w=w: x + w * (f(x) - x)) for c, f in EFFECTS[target].items()})
                    nr = G["df"].loc[i]; cat_after = nr["health"]
                    if RANK[cat_after] > RANK[actual]:
                        G["restored"] += 1; add_score(10); gain += 10; verdict += " · 🌟 region restored (+10)"
                table.append({"Region": f"{r['region_id']} {r['name']}", "You predicted": p.value if p.value in CLASSES else "–",
                              "ML model": ml_pred, "Actual": actual, "Strategy": s.value if s.value in STRATEGIES else "skipped",
                              "Outcome": verdict, "Health": f"{hs_before:.0f} → {G['df'].loc[i, 'health_score']:.0f}", "Pts": gain})
            refresh_hud()
            display(pd.DataFrame(table))
            display(note(f"🤖 The model matched the actual condition in <b>{pts_ml}/{len(affected)}</b> of these tough regions; you matched <b>{pts_pred}/{n_pred}</b> of your predictions. "
                         f"Regions restored to a better category: <b>{G['restored']}</b> · Ocean health now <b>{ocean_health():.1f}%</b>.", "info"))
            finish_mission(4, "Supervised machine learning & data-driven decisions",
                           "In <b>supervised learning</b> a model learns from labelled examples (the <b>training set</b>) and is judged on unseen data (the <b>test set</b>) using "
                           "<b>accuracy</b> and a <b>confusion matrix</b>. Its predictions support – but do not replace – careful reading of the data.")
            display(btn("🏆 See Final Report", lambda b: screen_final(), "warning", "260px"))
    with screen:
        head("🌱 Mission 4: Restore the Ocean Ecosystem", "Use machine learning and your data skills")
        display(note(f"🤖 A <b>Random Forest</b> was just trained on {ml['n_train']} synthetic samples and tested on {ml['n_test']} unseen samples "
                     f"(80/20 split) → <b>accuracy {ml['acc'] * 100:.1f}%</b>.", "good"))
        fig, ax = plt.subplots(1, 2, figsize=(12, 3.8))
        ConfusionMatrixDisplay(ml["cm"], display_labels=CLASSES).plot(ax=ax[0], cmap="Blues", colorbar=False)
        ax[0].set_title("Confusion matrix (test set)")
        ml["imp"].sort_values().rename(FEATURE_LABELS).plot.barh(ax=ax[1], color="#2a9fd6"); ax[1].set_title("What the model relies on (feature importance)")
        plt.tight_layout(); plt.show()
        summ = pd.DataFrame({"At start": d0["health"].value_counts(), "Now": d["health"].value_counts()}).reindex(CLASSES).fillna(0).astype(int)
        display(note(f"📋 <b>Report from missions 1–3:</b> animals rescued <b>{G['animals']}</b> · polluted regions cleaned <b>{G['cleaned']}</b> · "
                     f"ocean health <b>{d0['health_score'].mean():.1f}% → {ocean_health():.1f}%</b>. Regions per category:"))
        display(summ.T)
        if not affected:
            display(note("🎉 Every region is already Healthy!", "good")); finish_mission(4, "Supervised machine learning", "The model confirmed a healthy ocean.")
            display(btn("🏆 See Final Report", lambda b: screen_final(), "warning", "260px")); return
        display(note(f"🎯 <b>Objective:</b> these are the {len(affected)} most affected regions. For each, <b>predict</b> its condition (+10 if right), then choose the <b>restoration strategy</b> that "
                     f"fits its worst stressor (+30 best / +15 acceptable, +10 if the region improves a category). You have only <b>{MAX_ACTIONS}</b> restoration actions – leave the rest on skip."))
        for r in rows: display(r)
        display(go, out)

## Step 8 – Final report and animated victory screen

In [ ]:
CONCEPTS = ["Data exploration & sampling", "Classification with environmental data", "Correlation (not causation)",
            "Outlier detection (IQR / Z-score) & visualisation", "Supervised machine learning (accuracy, confusion matrix)", "Data-driven decision making"]

def screen_final():
    d0, d = G["df0"], G["df"]; hp = ocean_health(); acc = G["ml"]["acc"] * 100 if G["ml"] else float("nan")
    done = len(G["missions"]); composite = 0.5 * min(G["score"] / 550, 1) * 100 + 0.5 * hp
    rank = ("🥇 MARINE HERO" if composite >= 80 else "🥈 OCEAN GUARDIAN" if composite >= 62 else "🥉 REEF RANGER" if composite >= 45 else "🐠 DEEP-SEA ROOKIE")
    win = done == 4
    with screen:
        clear_output(wait=True); refresh_hud()
        display(widgets.HTML(banner("🏆 OCEAN RESTORED – YOU ARE A MARINE HERO! 🏆" if win else "📋 Mission Report",
                                    f"Final rank: {rank}" if win else "Finish all four missions to unlock the victory screen", height=230, victory=win)))
        display(hud_w)
        items = [("⭐ Total score", G["score"]), ("🎯 Missions completed", f"{done}/4"), ("🐢 Marine animals rescued", G["animals"]),
                 ("🧹 Polluted regions cleaned", G["cleaned"]), ("🌱 Regions restored (Mission 4)", G["restored"]),
                 ("🌊 Final ocean health", f"{hp:.1f}%  (started at {d0['health_score'].mean():.1f}%)"),
                 ("🤖 ML model accuracy", f"{acc:.1f}%" if G["ml"] else "not trained yet"), ("🏅 Final performance ranking", f"{rank}  (composite {composite:.0f}/100)")]
        rows = "".join(f"<tr><td style='padding:6px 14px;border-bottom:1px solid #cde'>{a}</td><td style='padding:6px 14px;border-bottom:1px solid #cde'><b>{b}</b></td></tr>" for a, b in items)
        display(widgets.HTML(f"<table style='font-family:Segoe UI,Arial;font-size:15px;color:#12324d;background:#f1faff;border-radius:10px'>{rows}</table>"))
        display(note("<b>📚 Data Science concepts you learned:</b><br>" + "<br>".join(f"✔️ {c}" for c in CONCEPTS), "concept"))
        order = d0.sort_values("health_score").index
        fig, ax = plt.subplots(figsize=(13, 3.4)); x = np.arange(len(order))
        ax.bar(x - 0.2, d0.loc[order, "health_score"], 0.4, label="Start", color="#e4572e")
        ax.bar(x + 0.2, d.loc[order, "health_score"], 0.4, label="Final", color="#2a9fd6")
        ax.set_xticks(x); ax.set_xticklabels(d0.loc[order, "region_id"], rotation=90, fontsize=7)
        ax.set_ylabel("Health score"); ax.set_title("Ocean health score per region: start vs final"); ax.legend(); plt.tight_layout(); plt.show()
        display(widgets.HBox([btn("🔄 Play again", lambda b: start_game(), "success", "180px"), btn("🏠 Main menu", lambda b: screen_menu(), "primary", "180px")]))

---
# 🚀 LAUNCH THE GAME
Run the next cell – the game appears right below it.

In [ ]:
new_game()
display(screen)
screen_menu()

---
# 🧪 PLAYTESTING AND TESTING

## A. Automated self-test (no clicking needed)
This checks that the dataset, outlier detection, statistics and machine-learning pipeline really work (no hard-coded results).

In [ ]:
def run_self_test():
    w = build_world()
    assert w.shape[0] == 36 and {"region_id", "temperature", "oxygen", "plastic", "water_quality", "animal_pop", "health"} <= set(w.columns)
    assert set(w["health"]) <= set(CLASSES) and w[FEATURES].notna().all().all()
    assert w["health"].nunique() == 3, "dataset should contain Healthy, Warning and Critical regions"
    thr_iqr, thr_z = outlier_threshold(w["plastic"], "IQR method", 1.5), outlier_threshold(w["plastic"], "Z-score method", 2.0)
    assert (w["plastic"] > thr_iqr).sum() >= 1, "IQR method should flag at least one outlier"
    assert outlier_threshold(w["plastic"], "IQR method", 3.0) > thr_iqr, "threshold must react to k"
    ml = train_model()
    assert 0.80 <= ml["acc"] <= 1.0 and ml["cm"].shape == (3, 3) and ml["cm"].sum() == ml["n_test"]
    pred = ml["model"].predict(w[FEATURES]); assert len(pred) == 36 and set(pred) <= set(CLASSES)
    corr = w[FEATURES].corr(); assert corr.loc["oxygen", "plastic"] < -0.3
    print(f"✅ dataset OK · IQR threshold {thr_iqr:.1f} flags {(w['plastic'] > thr_iqr).sum()} regions · Z threshold {thr_z:.1f} · "
          f"ML accuracy {ml['acc'] * 100:.1f}% · corr(oxygen, plastic) = {corr.loc['oxygen', 'plastic']:.2f}")
    print("All self-tests passed.")

run_self_test()

## B. Human playtest checklist
Play the game at least twice (once carelessly, once carefully) and tick each item.

| # | Test | Expected result | ✔ |
|---|---|---|---|
| 1 | Main menu buttons (Start / Instructions / Exit) | Each opens the right screen; *Back* returns to the menu | ☐ |
| 2 | Learning objectives shown before Mission 1 | 6 objectives visible | ☐ |
| 3 | Mission 2, 3 and 4 are locked at the start | Only Mission 1 is clickable on the map | ☐ |
| 4 | Mission 1 – arrow buttons | Submarine moves, oxygen drops 4% per move, score +10 per new region | ☐ |
| 5 | Mission 1 – oxygen reaches 0 | Arrows disable; *Finish exploring* still works | ☐ |
| 6 | Mission 1 – biased vs. spread-out sample | Different feedback + graph | ☐ |
| 7 | Mission 2 – decide for all 8 animals | Warning if any is undecided; feedback explains each decision | ☐ |
| 8 | Mission 2 – missed / wasted rescue | Ocean health drops / oxygen wasted | ☐ |
| 9 | Mission 3 – change method / k | Threshold and flagged regions update live | ☐ |
| 10 | Mission 3 – exceed 12 units | Extra region refuses to toggle on | ☐ |
| 11 | Mission 3 – before/after graphs | Bars shrink for cleaned regions; ocean health rises | ☐ |
| 12 | Mission 4 – model accuracy + confusion matrix | Shown, changes if you edit the dataset code | ☐ |
| 13 | Mission 4 – more than 6 actions | Warning appears, nothing is applied | ☐ |
| 14 | Final report | All 8 report lines + victory animation | ☐ |
| 15 | *Play again* | Score, health and missions reset | ☐ |

## C. Playtest log (fill in)
| Tester | Date | Difficulty (1-5) | Bugs found | Fix / improvement made |
|---|---|---|---|---|
|  |  |  |  |  |

**Improvements made after playtesting:** step-by-step buttons instead of keyboard events (works on every Jupyter platform), oxygen cost per move to prevent aimless exploring, cleanup budget to force strategy, and a hint legend of stress signals in Mission 2.

## D. Requirement coverage
| Requirement | Where |
|---|---|
| Pandas / NumPy synthetic dataset (Region ID, temperature, oxygen, plastic, water quality, population, health category) | Step 1 – `make_ocean`, `build_world` |
| Sampling, classification, correlation, IQR / Z-score, Random-Forest with train/test split, accuracy & confusion matrix | Steps 4–7 |
| Score, oxygen, ocean health, mission progress & unlocking, ranking | Step 2 HUD, Step 3 hub, Step 8 |
| Decisions change the data (`update_region`) and the score | Steps 4–7 |
| Final report + animated victory | Step 8 |
| Documentation + playtesting | Top and bottom of this notebook |